In [1]:
!pip install sentencepiece wandb fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.3-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.3-py3-none-any.whl (313 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653975 sha256=e62a5df8c38d810a018f2529c4e746400a64288385e35554240512fcde8b2b68
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [2]:
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('gdrive')

Mounted at gdrive


In [3]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: daria-kay000 (daria-kay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import torch
import wandb
import numpy as np
import os
from tqdm import tqdm
import fasttext
import polars as pl
from sklearn.metrics import f1_score, classification_report
from sentencepiece import SentencePieceTrainer, SentencePieceProcessor
from huggingface_hub import hf_hub_download

DATA_DIRECTORY = "gdrive/MyDrive/data/" if IS_COLAB else "../data/interim/"
INTERIM_FILES = "gdrive/MyDrive/data/" if IS_COLAB else "misc/"
ARANEUM_FASTTEXT = "araneum_none_fasttextcbow_300_5_2018"
WANDB_PROJECT = "expirements"
DEVICE = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
DEVICE

'cuda'

In [15]:
class CommentsDataset(torch.utils.data.Dataset):
    """
    Torch dataset for loading and tokenizing comments from a csv file
    """
    LABEL_MAPPING = {
        "NORMAL": 0,
        "INSULT": 1,
        "THREAT": 2,
        "OBSCENITY": 3
    }

    def __init__(self, filename: str, tokenizer):
        self.tokenizer = tokenizer
        data_sorted = (
            pl.read_csv(filename)
            .sort(pl.col("comment").str.len_chars())
            .with_columns(pl.col("label").replace_strict(self.LABEL_MAPPING).alias("label"))
        )
        self.comments = data_sorted["comment"]
        self.labels = data_sorted["label"]

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, idx):
        return self.tokenizer.encode(self.comments[idx]), self.labels[idx]

def pad_collate(batch: list[tuple[list[int], int]], pad_id: int):
    """
    Pad to maximum length in a batch
    """
    batch_tokens = []
    batch_labels = []
    max_len = 0
    for tokens, label in batch:
        if len(tokens) > max_len:
            max_len = len(tokens)
        batch_tokens.append(torch.tensor(tokens))
        batch_labels.append(label)
    padded = [torch.nn.functional.pad(tokens, (0, max_len - len(tokens)), value=pad_id) for tokens in batch_tokens]
    return torch.vstack(padded), torch.tensor(batch_labels)

def print_report(model, tokenizer, test_dataset):
    predicted_labels = []
    true_labels = []
    model.eval()
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, collate_fn=lambda batch: pad_collate(batch, tokenizer.pad_id()))
    with torch.inference_mode():
        for X, labels in test_dataloader:
            X, labels = X.to(device=DEVICE), labels.to(device=DEVICE)
            predicted_labels.extend(model(X).argmax(axis=1).cpu().tolist())
            true_labels.extend(labels.cpu().tolist())
    print(classification_report(true_labels, predicted_labels))

def train(
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        criterion: torch.nn.Module,
        train_dataset: CommentsDataset,
        val_dataset: CommentsDataset,
        pad_id: int,
        n_epoch: int = 3,
        batch_size: int = 512,
        run_config: dict = None,
):
    # initialize the dataloaders
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        collate_fn=lambda batch: pad_collate(batch, pad_id)
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        collate_fn=lambda batch: pad_collate(batch, pad_id)
    )

    # set up a wandb run
    wandb_run_name = None
    if run_config:
        wandb_run_name = run_config.get("name", None)
        if wandb_run_name:
            del run_config["name"]
        run_config["batch_size"] = batch_size
        run_config["n_epoch"] = n_epoch
        run_config["optimizer"] = optimizer.__class__.__name__
        run_config["lr"] = optimizer.param_groups[0]["lr"]

    # start training loop
    with wandb.init(project=WANDB_PROJECT, name=wandb_run_name, config=run_config) as wandb_run, tqdm(total=len(train_dataloader) * n_epoch) as pbar:
        global_step = 0
        wandb_run.define_metric("global_step") # required to make all graphics share the same x axis
        wandb_run.define_metric("*", step_metric="global_step")
        wandb_run.watch(model, log="all", log_freq=100)

        epoch_val_loss = 0
        epoch_train_loss = 0
        epoch_f1 = 0
        for epoch in range(n_epoch):
            update_epoch_metrics = True
            train_cum_loss = 0
            for X, labels in train_dataloader:
                model.train()
                X, labels = X.to(device=DEVICE), labels.to(device=DEVICE)
                optimizer.zero_grad()
                pred = model(X)
                loss = criterion(pred, labels)
                train_cum_loss += loss.item() * X.size(0)
                loss.backward()
                optimizer.step()

                if update_epoch_metrics:
                    epoch_train_loss = train_cum_loss / len(train_dataset)

                    with torch.inference_mode():
                        val_cum_loss = 0
                        predicted_labels = []
                        true_labels = []
                        for X, labels in val_dataloader:
                            model.eval()
                            X, labels = X.to(device=DEVICE), labels.to(device=DEVICE)
                            predicted = model(X)
                            val_cum_loss += criterion(predicted, labels).item() * labels.size(0)
                            predicted_labels.extend(predicted.argmax(axis=1).cpu().tolist())
                            true_labels.extend(labels.cpu().tolist())
                        epoch_val_loss =  val_cum_loss / len(val_dataset)
                        epoch_f1 = f1_score(true_labels, predicted_labels, average="macro")
                    update_epoch_metrics = False

                wandb_run.log({
                    "global_step": global_step,
                    "epoch/train-loss": epoch_train_loss,
                    "epoch/val-loss": epoch_val_loss,
                    "epoch/f1-score": epoch_f1,
                    "batch/train-loss": loss.item()
                })
                global_step += 1
                pbar.update()
        wandb_run.unwatch(model)

def build_embedding_matrix(tokenizer, fasttext_model) -> torch.Tensor:
    vocab_dim = tokenizer.vocab_size()
    embed_dim = fasttext_model.get_dimension()
    # task specific tokens is going to be initialized random then learned
    embedding_params = np.random.normal(
        loc=0,
        scale= 2 / (vocab_dim + embed_dim), # variance control
        size=(vocab_dim, embed_dim),
    )
    out_of_fasttext_vocab = 0
    for idx in range(vocab_dim):
        if idx == tokenizer.pad_id():
            embedding_params[idx] = np.zeros(embed_dim)
        token = tokenizer.decode(idx)
        if token in fasttext_model:
            embedding_params[idx] = fasttext_model[token]
        else:
            out_of_fasttext_vocab += 1
    print(f"Out of fasttext vocab rate: {(out_of_fasttext_vocab / vocab_dim):.3f}")
    return torch.tensor(embedding_params, dtype=torch.float32)

# BPE tokenizer trained on train comments
if not os.path.isfile(INTERIM_FILES + "bpe.model"):
    SentencePieceTrainer.train(
        sentence_iterator=iter(pl.read_csv(DATA_DIRECTORY + "train.csv")["comment"]),
        model_prefix=INTERIM_FILES + "bpe",
        model_type="bpe",
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3
    )
bpe_tokenizer = SentencePieceProcessor(model_file=INTERIM_FILES + "bpe.model")
bpe_tokenized_train = CommentsDataset(DATA_DIRECTORY + "train.csv", bpe_tokenizer)
bpe_tokenized_test = CommentsDataset(DATA_DIRECTORY + "test.csv", bpe_tokenizer)

# unigram tokenizer trained on train comments
if not os.path.isfile(INTERIM_FILES + "unigram.model"):
    SentencePieceTrainer.train(
        sentence_iterator=iter(pl.read_csv(DATA_DIRECTORY + "train.csv")["comment"]),
        model_prefix=INTERIM_FILES + "unigram",
        model_type="unigram",
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3
    )
unigram_tokenizer = SentencePieceProcessor(model_file=INTERIM_FILES + "unigram.model")
unigram_tokenized_train = CommentsDataset(DATA_DIRECTORY + "train.csv", unigram_tokenizer)
unigram_tokenized_test = CommentsDataset(DATA_DIRECTORY + "test.csv", unigram_tokenizer)

In [6]:
fasttext_model = fasttext.load_model(
    hf_hub_download(repo_id="facebook/fasttext-ru-vectors", filename="model.bin")
)
bpe_fasttext_embeddings = build_embedding_matrix(bpe_tokenizer, fasttext_model)
unigram_fasttext_embeddings = build_embedding_matrix(unigram_tokenizer, fasttext_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.bin:   0%|          | 0.00/7.26G [00:00<?, ?B/s]

Out of fasttext vocab rate: 0.063
Out of fasttext vocab rate: 0.060


## GRU + предоубученные эмбеддинги fasttext

In [7]:
class PretrainedEmbeddingGRUClf(torch.nn.Module):

    def __init__(self, embed_matrix: torch.Tensor, padding_idx, hidden_dim: int, bidirectional: bool, num_layers: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        vocab_dim, embed_dim = embed_matrix.size()
        self.embedding = torch.nn.Embedding.from_pretrained(embed_matrix.float(), freeze=False, padding_idx=padding_idx)
        self.rnn = torch.nn.GRU(
            batch_first=True,
            input_size=embed_dim,
            hidden_size=hidden_dim,
            bidirectional=bidirectional,
            num_layers=num_layers
        )
        self.linear = torch.nn.Linear(
            in_features=2 * hidden_dim if bidirectional else hidden_dim,
            out_features=4
        )

    def forward(self, input):
        emb_output = self.embedding(input)
        rnn_output, _ = self.rnn(emb_output)
        return self.linear(rnn_output[:, -1, :])

### BPE

In [18]:
fasttext_gru_bpe = PretrainedEmbeddingGRUClf(bpe_fasttext_embeddings, bpe_tokenizer.pad_id(), hidden_dim=300, bidirectional=True, num_layers=3)
fasttext_gru_bpe.to(device=DEVICE)
optim = torch.optim.AdamW(fasttext_gru_bpe.parameters(), lr=5e-4, fused=True)
criterion = torch.nn.CrossEntropyLoss()
config = {"name": "Fasttext + GRU"}

train(fasttext_gru_bpe, optim, criterion, bpe_tokenized_train, bpe_tokenized_test, bpe_tokenizer.pad_id(), run_config=config, n_epoch=5)
print_report(fasttext_gru_bpe, bpe_tokenizer, bpe_tokenized_test)

100%|██████████| 1940/1940 [01:00<00:00, 32.27it/s]


batch/train-loss,█▇▇▇▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▂▂▂▁▂▁▂▂▂▂▂▂▁▁▁▁▁▂▂▂
epoch/f1-score,▁▁▁▁▁▁▁▁▁▁▇▇▇▇▇▇▇███████████████████████
epoch/train-loss,██████▄▄▄▄▄▄▄▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val-loss,█████████▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇██████
batch/train-loss,0.09894
epoch/f1-score,0.84741
epoch/train-loss,0.00027
epoch/val-loss,0.14019
global_step,1939


              precision    recall  f1-score   support

           0       0.98      0.98      0.98     40736
           1       0.84      0.82      0.83      5713
           2       0.83      0.83      0.83      2356
           3       0.70      0.79      0.74       852

    accuracy                           0.95     49657
   macro avg       0.84      0.86      0.85     49657
weighted avg       0.95      0.95      0.95     49657



### Unigram


In [19]:
fasttext_gru_unigram = PretrainedEmbeddingGRUClf(unigram_fasttext_embeddings, unigram_tokenizer.pad_id(), hidden_dim=300, bidirectional=True, num_layers=3)
fasttext_gru_unigram.to(device=DEVICE)
optim = torch.optim.AdamW(fasttext_gru_unigram.parameters(), lr=5e-4, fused=True)
criterion = torch.nn.CrossEntropyLoss()
config = {"name": "Fasttext + GRU + unigram"}

train(fasttext_gru_unigram, optim, criterion, unigram_tokenized_train, unigram_tokenized_test, unigram_tokenizer.pad_id(), run_config=config, n_epoch=5)
print_report(fasttext_gru_unigram, unigram_tokenizer, unigram_tokenized_test)

100%|██████████| 1940/1940 [00:47<00:00, 40.96it/s]


batch/train-loss,▇█▇▇▅▆▇▄▆▆▂▂▃▃▃▄▂▂▂▁▁▂▁▂▂▄▄▃▂▁▂▂▁▁▂▁▂▃▁▃
epoch/f1-score,▁▁▁▁▁▁▁▁▁▇▇▇▇▇██████████████████████████
epoch/train-loss,█████████▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val-loss,█████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
global_step,▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█
batch/train-loss,0.10969
epoch/f1-score,0.83111
epoch/train-loss,0.00029
epoch/val-loss,0.1737
global_step,1939


              precision    recall  f1-score   support

           0       0.98      0.97      0.98     40736
           1       0.83      0.78      0.81      5713
           2       0.70      0.90      0.79      2356
           3       0.65      0.79      0.71       852

    accuracy                           0.94     49657
   macro avg       0.79      0.86      0.82     49657
weighted avg       0.95      0.94      0.94     49657



## Bidirectional RNN + предоубученные эмбеддинги fasttext



In [20]:
class PretrainedEmbeddingRNNClf(torch.nn.Module):

    def __init__(self, embed_matrix: torch.Tensor, padding_idx, hidden_dim: int, bidirectional: bool, num_layers: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        vocab_dim, embed_dim = embed_matrix.size()
        self.embedding = torch.nn.Embedding.from_pretrained(embed_matrix.float(), freeze=False, padding_idx=padding_idx)
        self.rnn = torch.nn.RNN(
            batch_first=True,
            input_size=embed_dim,
            hidden_size=hidden_dim,
            bidirectional=bidirectional,
            num_layers=num_layers,
            nonlinearity="relu"
        )
        self.linear = torch.nn.Linear(
            in_features=2 * hidden_dim if bidirectional else hidden_dim,
            out_features=4
        )

    def forward(self, input):
        emb_output = self.embedding(input)
        rnn_output, _ = self.rnn(emb_output)
        return self.linear(rnn_output[:, -1, :])

### BPE токенизация, обученная на трейне

*   эмбеддинги дали прирост метрики, но они не практически не дообучаются
*   в то же время ошибка на трейне и на тесте все равно сильно отличается

In [21]:
bpe_fasttext_rnn = PretrainedEmbeddingRNNClf(bpe_fasttext_embeddings, bpe_tokenizer.pad_id(), hidden_dim=200, bidirectional=True, num_layers=4)
bpe_fasttext_rnn.to(device=DEVICE)
optim = torch.optim.AdamW(bpe_fasttext_rnn.parameters(), lr=1e-4, fused=True)
criterion = torch.nn.CrossEntropyLoss()
config = {"name": "Fasttext + RNN"}

train(bpe_fasttext_rnn, optim, criterion, bpe_tokenized_train, bpe_tokenized_test, bpe_tokenizer.pad_id(), run_config=config, n_epoch=10)
print_report(bpe_fasttext_rnn, bpe_tokenizer, bpe_tokenized_test)

100%|██████████| 3880/3880 [01:32<00:00, 42.01it/s]


batch/train-loss,█▆▃▃▄▃▃▃▂▄▂▄▃▂▅▄▁▂▂▂▁▁▃▃▂█▂▂▂▃▂▂▃▁▁▃▁▁▂▂
epoch/f1-score,▁▁▁▁▂▂▂▂▂▄▄▄▄▄▄▆▆▆▆▆▆▆▆▆▆▆▄▄▄▆▆▆▆▆▆█████
epoch/train-loss,█████▇▇▇▇▇▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▃▃▂▂▂▂▁
epoch/val-loss,█████▇▇▇▇▇▇▄▄▂▂▂▂▂▂▂▁▁▁▁▁▁▅▅▅▅▅▃▃▃▃▃▁▁▁▁
global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
batch/train-loss,0.21762
epoch/f1-score,0.59452
epoch/train-loss,0.0004
epoch/val-loss,0.25662
global_step,3879


              precision    recall  f1-score   support

           0       0.96      1.00      0.98     40736
           1       0.77      0.74      0.76      5713
           2       0.87      0.67      0.76      2356
           3       0.14      0.00      0.00       852

    accuracy                           0.93     49657
   macro avg       0.69      0.60      0.62     49657
weighted avg       0.92      0.93      0.92     49657



### Unigram токенизация обученная на трейне

In [22]:
unigram_fasttext_rnn = PretrainedEmbeddingRNNClf(
    unigram_fasttext_embeddings,
    unigram_tokenizer.pad_id(),
    hidden_dim=200,
    bidirectional=True,
    num_layers=4
)
unigram_fasttext_rnn.to(device=DEVICE)
optim = torch.optim.AdamW(unigram_fasttext_rnn.parameters(), lr=1e-4, fused=True)
criterion = torch.nn.CrossEntropyLoss()
config = {"name": "Fasttext + RNN + unigram tokenizer"}

train(unigram_fasttext_rnn, optim, criterion, unigram_tokenized_train, unigram_tokenized_test, unigram_tokenizer.pad_id(), run_config=config, n_epoch=10)
print_report(unigram_fasttext_rnn, unigram_tokenizer, unigram_tokenized_test)

100%|██████████| 3880/3880 [01:05<00:00, 58.88it/s]


batch/train-loss,▇▆▆▅█▂▂▃▂▄▃▂▃▂▂▂▁▂▁▁▃▂▂▂▃▂▃▂▂▂▂▂▃▂▁▂▂▂▁▃
epoch/f1-score,▄▄▄▄▄▄▄▄▄▄▄▁▁▁▁▄▄▄▄███▇▇▇▇▇▇▇▇▇▇██████▆▆
epoch/train-loss,▆▆▆▆▆▆▆▆▆████▄▄▄▂▂▂▂▂▂▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val-loss,▆▆▆▅▅▅▅▅▅██████▃▃▃▃▃▁▁▁▁▁▁▂▂▂▂▂▂▁▁▁▂▂▂▂▂
global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
batch/train-loss,0.30991
epoch/f1-score,0.30655
epoch/train-loss,0.00069
epoch/val-loss,0.53194
global_step,3879


              precision    recall  f1-score   support

           0       0.95      1.00      0.97     40736
           1       0.59      0.71      0.65      5713
           2       0.00      0.00      0.00      2356
           3       0.62      0.01      0.01       852

    accuracy                           0.90     49657
   macro avg       0.54      0.43      0.41     49657
weighted avg       0.86      0.90      0.87     49657



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Bidirectional RNN + встроенные эбмеддинги

*   сильное переобучение
*   эмбеддинги не обучаются (градиенты нулевые) -- возможно из-за BPTT вспомнить как работает
*    мало данных, чтобы выучивался редкий класс obcsenity


In [23]:
class BasicRNNClf(torch.nn.Module):

    def __init__(self, vocab_dim: int, padding_idx: int, embed_dim: int, hidden_dim: int, bidirectional: bool, num_layers: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_dim, embedding_dim=embed_dim, padding_idx=padding_idx)
        self.rnn = torch.nn.RNN(
            batch_first=True,
            input_size=embed_dim,
            hidden_size=hidden_dim,
            bidirectional=bidirectional,
            num_layers=num_layers,
            nonlinearity="relu"
        )
        self.linear = torch.nn.Linear(
            in_features=2 * hidden_dim if bidirectional else hidden_dim,
            out_features=4
        )

    def forward(self, input):
        emb_output = self.embedding(input)
        rnn_output, _ = self.rnn(emb_output)
        return self.linear(rnn_output[:, -1, :])


In [24]:
bpe_rnn = BasicRNNClf(bpe_tokenizer.vocab_size(), bpe_tokenizer.pad_id(), embed_dim=300, hidden_dim=200, bidirectional=True, num_layers=4)
bpe_rnn.to(device=DEVICE)
optim = torch.optim.AdamW(bpe_rnn.parameters(), lr=1e-4, fused=True)
criterion = torch.nn.CrossEntropyLoss()
config = {"name": "Vanila RNN"}

train(bpe_rnn, optim, criterion, bpe_tokenized_train, bpe_tokenized_test, bpe_tokenizer.pad_id(), run_config=config, n_epoch=10)
print_report(bpe_rnn, bpe_tokenizer, bpe_tokenized_test)

100%|██████████| 3880/3880 [01:32<00:00, 42.15it/s]


batch/train-loss,▆██▇▆▃▂▃▃▄▄▅▅▃▂▂▁▁▂▄▁▁▁▂▁█▃▂▁▁▁▂▄▃▁▁▁▁▅▄
epoch/f1-score,▁▁▁▁▅▅▅▅▆▆▃▃▃▃▃▃██████████████▅▅▅███████
epoch/train-loss,███████▂▃▃▃▃▃▃▃▆▆▆▆▆▃▃▃▃▃▃▂▂▂▁▁▁▁▂▂▃▃▃▃▃
epoch/val-loss,█████▃▃▃▃▃▂▂▂▆▆▆▆▆▆▆▂▂▂▂▂▁▁▁▂▂▂▂▂▁▁▁▁▂▂▂
global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇█████
batch/train-loss,1.50287
epoch/f1-score,0.36016
epoch/train-loss,0.00167
epoch/val-loss,0.51868
global_step,3879


              precision    recall  f1-score   support

           0       0.93      0.99      0.96     40736
           1       0.53      0.57      0.55      5713
           2       0.00      0.00      0.00      2356
           3       0.00      0.00      0.00       852

    accuracy                           0.88     49657
   macro avg       0.36      0.39      0.38     49657
weighted avg       0.82      0.88      0.85     49657



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
